# Local T2I-CompBench Official Evaluation

This notebook evaluates only the generated 100-prompt subset already present in `data/*.zip`. It extracts those method outputs into the repo's benchmark run layout, pins the official T2I-CompBench checkout, verifies the BLIP VQA and UniDet/Detectron2 local model paths on GPU 0, then runs the official color, shape, texture, and 2D spatial evaluators.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "aim_flow").exists() and (path / "configs" / "t2i_compbench_100_seed13.json").exists():
            return path
    raise RuntimeError("Could not find the aim-flow repo root from the current notebook location.")


REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)

SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
METHODS = ["spfc", "rectified_cfgpp", "cfg", "base"]
CATEGORIES = ["color", "shape", "texture", "spatial"]
ZIP_FILES = {
    "spfc": "spfc.zip",
    "rectified_cfgpp": "rectifiedcfg++.zip",
    "cfg": "cfg.zip",
    "base": "base.zip",
}

MANIFEST_PATH = REPO_DIR / "configs" / "t2i_compbench_100_seed13.json"
DATA_DIR = REPO_DIR / "data"
RUN_ROOT = REPO_DIR / "benchmarks" / "runs" / "local_t2i_compbench_seed13"
REPORT_DIR = REPO_DIR / "benchmarks" / "reports" / "local_t2i_compbench_seed13"
EVAL_DIR = REPORT_DIR / "eval"
T2I_REPO_DIR = REPO_DIR / "external" / "T2I-CompBench"
T2I_COMPBENCH_COMMIT = "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"
ENV_PREFIX = REPO_DIR / ".venv" / "t2i-compbench-py310"
EVAL_PYTHON = ENV_PREFIX / "bin" / "python"
CONDA_PKGS_DIR = REPO_DIR / ".conda_pkgs"
CUDA_VISIBLE_DEVICES = "0"


def run_cmd(cmd, cwd: Path = REPO_DIR, env: dict | None = None) -> None:
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items()})
    subprocess.run(cmd, cwd=str(cwd), env=merged_env, check=True)


def eval_env(extra: dict | None = None) -> dict:
    env = {
        "CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES,
        "CUDA_HOME": str(ENV_PREFIX),
        "PATH": f"{ENV_PREFIX / 'bin'}:{os.environ.get('PATH', '')}",
    }
    if extra:
        env.update(extra)
    return env


print("Repo:", REPO_DIR)
print("Manifest:", MANIFEST_PATH)
print("Run root:", RUN_ROOT)
print("Official repo:", T2I_REPO_DIR)
print("Evaluator Python:", EVAL_PYTHON)

## Extract The Provided Subset

This cell extracts only the PNGs whose sample ids are present in `configs/t2i_compbench_100_seed13.json`. It does not download or prepare the full T2I-CompBench dataset.

In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

samples = manifest["samples"]
expected_ids = {sample["id"] for sample in samples}
categories = sorted({sample["category"] for sample in samples})
print(f"Manifest subset: {len(samples)} samples across {categories}")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
extracted = {}
for method in METHODS:
    zip_path = DATA_DIR / ZIP_FILES[method]
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    target_dir = RUN_ROOT / "t2i_compbench" / method
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        members = []
        zip_methods = set()
        for info in zf.infolist():
            parts = Path(info.filename).parts
            if info.is_dir() or len(parts) != 5:
                continue
            if parts[0] != RUN_SLUG or parts[1] != "runs" or parts[2] != "t2i_compbench":
                continue
            if not parts[4].endswith(".png"):
                continue
            zip_methods.add(parts[3])
            sample_id = Path(parts[4]).stem
            if sample_id in expected_ids:
                members.append((info, sample_id))
        found_ids = {sample_id for _, sample_id in members}
        missing = sorted(expected_ids - found_ids)
        extra_methods = sorted(zip_methods)
        if missing:
            raise RuntimeError(f"{zip_path.name} is missing {len(missing)} manifest ids, first: {missing[:5]}")
        if len(found_ids) != len(expected_ids):
            raise RuntimeError(f"{zip_path.name} had duplicate or unexpected subset ids")
        for info, sample_id in members:
            out_path = target_dir / f"{sample_id}.png"
            with zf.open(info) as src, out_path.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    extracted[method] = len(list(target_dir.glob("*.png")))
    print(f"{method}: extracted {extracted[method]} images from {zip_path.name} (zip method dirs: {extra_methods})")

if any(count != len(expected_ids) for count in extracted.values()):
    raise RuntimeError(f"Extraction count mismatch: {extracted}")

## Pin The Official T2I-CompBench Checkout

In [ ]:
if not T2I_REPO_DIR.exists():
    T2I_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "https://github.com/Karine-Huang/T2I-CompBench.git", T2I_REPO_DIR])

run_cmd(["git", "-C", T2I_REPO_DIR, "fetch", "origin", T2I_COMPBENCH_COMMIT])
run_cmd(["git", "-C", T2I_REPO_DIR, "checkout", "--force", T2I_COMPBENCH_COMMIT])

required_scripts = [
    T2I_REPO_DIR / "BLIPvqa_eval" / "BLIP_vqa.py",
    T2I_REPO_DIR / "UniDet_eval" / "2D_spatial_eval.py",
]
for path in required_scripts:
    if not path.exists():
        raise FileNotFoundError(path)
print("Official T2I-CompBench commit:", subprocess.check_output(["git", "-C", str(T2I_REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

## Ensure The Local Official Evaluator Environment

The official spatial evaluator depends on Detectron2/UniDet, so this uses a Python 3.10 conda env with PyTorch CUDA 11.7 and a local Detectron2 source build. If the env already imports `torch`, `spacy`, and `detectron2`, this cell skips the setup.

In [ ]:
def official_env_imports_ok() -> bool:
    if not EVAL_PYTHON.exists():
        return False
    code = """
import torch, detectron2, spacy
assert torch.cuda.is_available(), 'CUDA is not available to the evaluator env'
spacy.load('en_core_web_sm')
print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))
print('detectron2', getattr(detectron2, '__version__', 'unknown'))
"""
    result = subprocess.run(
        [str(EVAL_PYTHON), "-c", code],
        cwd=str(REPO_DIR),
        env={**os.environ, **eval_env()},
        text=True,
    )
    return result.returncode == 0


if official_env_imports_ok():
    print("Evaluator env is ready; skipping install/build.")
else:
    conda = shutil.which("conda")
    if not conda:
        raise RuntimeError("conda is required to create the local official evaluator env")
    CONDA_PKGS_DIR.mkdir(parents=True, exist_ok=True)
    conda_env = {"CONDA_PKGS_DIRS": str(CONDA_PKGS_DIR)}
    if not EVAL_PYTHON.exists():
        run_cmd([conda, "create", "-y", "-p", ENV_PREFIX, "python=3.10", "pip"], env=conda_env)

    run_cmd([EVAL_PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run_cmd([EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir", "torch==2.0.1", "torchvision==0.15.2", "--index-url", "https://download.pytorch.org/whl/cu117"])
    run_cmd([
        EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir",
        "numpy==1.25.0", "Pillow==9.5.0", "opencv-python-headless==4.7.0.72",
        "spacy==3.5.3", "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.5.0/en_core_web_sm-3.5.0-py3-none-any.whl",
        "accelerate==0.17.0", "fairscale==0.4.4", "timm==0.4.12", "transformers==4.30.2",
        "ruamel.yaml==0.17.32", "pycocotools==2.0.6", "yacs==0.1.8", "fvcore==0.1.5.post20221221",
        "iopath==0.1.9", "omegaconf==2.3.0", "hydra-core==1.3.2", "pandas==2.0.2",
        "tqdm==4.65.0", "ftfy==6.1.1", "regex==2023.6.3", "matplotlib==3.7.1",
        "tabulate==0.9.0", "cloudpickle==2.2.1", "fire==0.5.0", "safetensors==0.3.1",
        "ninja==1.13.0", "setuptools<70",
    ])
    run_cmd([conda, "install", "-y", "-p", ENV_PREFIX, "gxx_linux-64=11", "gcc_linux-64=11"], env=conda_env)
    run_cmd([
        conda, "install", "-y", "-p", ENV_PREFIX,
        "-c", "nvidia/label/cuda-11.7.1", "-c", "nvidia",
        "cuda-nvcc=11.7.99", "cuda-cccl=11.7.91", "cuda-cudart=11.7.99", "cuda-cudart-dev=11.7.99",
        "cuda-driver-dev=11.7.99", "cuda-nvrtc=11.7.99", "cuda-nvrtc-dev=11.7.99",
        "libcublas=11.10.3.66", "libcublas-dev=11.10.3.66", "libcusparse=11.7.4.91", "libcusparse-dev=11.7.4.91",
        "libcusolver=11.4.0.1", "libcusolver-dev=11.4.0.1", "cuda-version=11.7",
    ], env=conda_env)

    build_env = eval_env({
        "CC": str(ENV_PREFIX / "bin" / "x86_64-conda-linux-gnu-gcc"),
        "CXX": str(ENV_PREFIX / "bin" / "x86_64-conda-linux-gnu-g++"),
        "CPATH": f"{ENV_PREFIX / 'include'}:{os.environ.get('CPATH', '')}",
        "CFLAGS": f"-I{ENV_PREFIX / 'include'}",
        "CXXFLAGS": f"-I{ENV_PREFIX / 'include'}",
        "TORCH_CUDA_ARCH_LIST": "7.5",
        "FORCE_CUDA": "1",
        "MAX_JOBS": "2",
    })
    run_cmd([
        EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--no-build-isolation", "--no-deps",
        "git+https://github.com/facebookresearch/detectron2.git@5aeb252b194b93dc2879b4ac34bc51a31b5aee13",
    ], env=build_env)
    if not official_env_imports_ok():
        raise RuntimeError("Evaluator env setup completed, but imports still failed.")

## Download UniDet RS200 Weights

In [ ]:
UNIDET_WEIGHT = T2I_REPO_DIR / "UniDet_eval" / "experts" / "expert_weights" / "Unified_learned_OCIM_RS200_6x+2x.pth"
UNIDET_WEIGHT_URL = "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth"
UNIDET_WEIGHT.parent.mkdir(parents=True, exist_ok=True)
if not UNIDET_WEIGHT.exists() or UNIDET_WEIGHT.stat().st_size < 100_000_000:
    tmp = UNIDET_WEIGHT.with_suffix(".pth.tmp")
    if tmp.exists():
        tmp.unlink()
    run_cmd(["curl", "-L", "--fail", "--retry", "3", "--connect-timeout", "30", UNIDET_WEIGHT_URL, "-o", tmp])
    tmp.replace(UNIDET_WEIGHT)
print("UniDet RS200 weight:", UNIDET_WEIGHT, UNIDET_WEIGHT.stat().st_size, "bytes")

## Smoke Test Official BLIP And UniDet Paths

In [ ]:
blip_smoke = r'''
import gc
import torch
from models.blip_vqa import blip_vqa
print('cuda available:', torch.cuda.is_available())
model = blip_vqa(
    pretrained='https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth',
    image_size=480,
    vit='base',
    vit_grad_ckpt=False,
    vit_ckpt_layer=0,
)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
print('BLIP loaded on', next(model.parameters()).device)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''
run_cmd([EVAL_PYTHON, "-c", blip_smoke], cwd=T2I_REPO_DIR / "BLIPvqa_eval", env=eval_env())

unidet_smoke = r'''
import gc
import torch
import detectron2
from experts.model_bank import load_expert_model
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('detectron2', getattr(detectron2, '__version__', 'unknown'))
model, transform = load_expert_model(task='obj_detection', ckpt='RS200')
print('UniDet model loaded:', type(model).__name__)
print('Transform:', transform)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''
run_cmd([EVAL_PYTHON, "-c", unidet_smoke], cwd=T2I_REPO_DIR / "UniDet_eval", env=eval_env())

## Run The Official Metrics On The Local Subset

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
run_cmd([
    EVAL_PYTHON, "scripts/bench_evaluate.py",
    "--benchmark", "t2i_compbench",
    "--manifest", MANIFEST_PATH,
    "--run-root", RUN_ROOT,
    "--methods", *METHODS,
    "--output-dir", EVAL_DIR,
    "--t2i-repo-dir", T2I_REPO_DIR,
    "--t2i-categories", *CATEGORIES,
    "--execute-official",
], cwd=REPO_DIR, env=eval_env())

score_path = EVAL_DIR / "t2i_compbench_scores.json"
with score_path.open("r", encoding="utf-8") as f:
    score_data = json.load(f)
print(json.dumps(score_data["scores"], indent=2, sort_keys=True))

## Build Report Tables And A Qualitative Grid

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
run_cmd([
    EVAL_PYTHON, "scripts/bench_report.py",
    "--t2i-scores", EVAL_DIR / "t2i_compbench_scores.json",
    "--output-dir", REPORT_DIR,
    "--run-root", RUN_ROOT,
    "--qualitative-manifest", MANIFEST_PATH,
    "--qualitative-output", REPORT_DIR / "qualitative_grid.png",
    "--qualitative-methods", *METHODS,
    "--qualitative-max-prompts", "8",
], cwd=REPO_DIR, env=eval_env())

print("Scores:", EVAL_DIR / "t2i_compbench_scores.json")
print("Report dir:", REPORT_DIR)
print("Qualitative grid:", REPORT_DIR / "qualitative_grid.png")